# 04 · Modelo CNN con Transfer Learning (solo imágenes)

**Objetivo:** Clasificar imágenes dermatoscópicas usando un backbone **MobileNetV3Large preentrenado en ImageNet**, en lugar de entrenar una CNN desde cero.

**Datos de entrada:** `../data/raw/hnmist_28_28_RGB.csv`, `../data/raw/HAM10000_metadata.csv`

**Resultado esperado:** Modelo guardado en `../models/cnn_model.keras` con ~85-90% de accuracy en test.

**Arquitectura:** MobileNetV3Large (backbone) + `Resizing(28→224)` + `GlobalAveragePooling2D` + cabeza densa.

**Estrategia:** entrenamiento en dos fases — primero solo la cabeza, luego fine-tuning del backbone.

## ¿Por qué Transfer Learning?

La CNN entrenada desde cero en imágenes 28×28 alcanza ~74% de accuracy. El problema es que con imágenes tan pequeñas y pocas muestras por clase, la red no tiene suficiente información visual para aprender features discriminativas.

**Transfer Learning** resuelve esto reutilizando un backbone que ya sabe detectar bordes, texturas y formas, porque fue entrenado en 14 millones de imágenes de ImageNet.

### Estrategia de dos fases

| Fase | Qué se entrena | Learning rate | Por qué |
|---|---|---|---|
| **1. Cabeza** | Solo las capas Dense nuevas | 1e-3 | El backbone ya tiene buenos pesos; la cabeza empieza aleatoria y necesita converger primero |
| **2. Fine-tuning** | Cabeza + último 30% del backbone | 1e-5 | Ajuste fino del backbone a nuestro dominio (dermatología ≠ ImageNet); LR muy pequeño para no destruir los pesos preentrenados |

### Arquitectura

```
Input(28,28,3)
    ↓
Resizing(224,224)          ← upscale dentro del modelo (sin tocar los datos)
    ↓
MobileNetV3Large (frozen)  ← 5.4M parámetros preentrenados, no se actualizan en fase 1
    ↓
GlobalAveragePooling2D     ← colapsa el mapa de features espacial a un vector
    ↓
Dense(512) + Dropout(0.4)
    ↓
Dense(128)
    ↓
Dense(7, softmax)          ← 7 clases diagnósticas
```

## Carga de datos y entrenamiento

El preprocesado (split 70/15/15, alineamiento por image_id, normalización per-canal) está centralizado en `utils.py`.

**Hiperparámetros clave:**
- `Resizing(224, 224)`: MobileNetV3Large fue entrenado con imágenes 224×224; el upscale se hace como capa Keras, no sobre los datos.
- `base.trainable = False`: congela los ~5.4M parámetros del backbone en la fase 1.
- `EarlyStopping(patience=10, restore_best_weights=True)`: misma política que en los otros modelos.
- LR fase 1 = `1e-3`: la cabeza empieza desde cero y necesita converger rápido.
- LR fase 2 = `1e-5`: fine-tuning muy conservador para no degradar los pesos preentrenados.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.layers import Input, Resizing, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# ── Carga de datos centralizados en utils.py ──────────────────────────────────
from utils import get_all_splits

(X_img_train, X_img_val, X_img_test,
 _,            _,          _,
 y_train, y_val, y_test, le) = get_all_splits()

X_train, X_val, X_test = X_img_train, X_img_val, X_img_test
num_classes = y_train.shape[1]

# ── Backbone preentrenado ──────────────────────────────────────────────────────
# include_top=False: descartamos la cabeza de clasificación de ImageNet
# input_shape=(224,224,3): el Resizing layer se encarga del upscale dentro del modelo
base = MobileNetV3Large(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False  # Fase 1: backbone completamente congelado

# ── Arquitectura ──────────────────────────────────────────────────────────────
inputs = Input(shape=(28, 28, 3))
x = Resizing(224, 224)(inputs)         # upscale al tamaño esperado por MobileNet
x = base(x, training=False)            # training=False: BatchNorm en modo inferencia
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(num_classes, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ── Fase 1: entrenar solo la cabeza ───────────────────────────────────────────
print("\n=== FASE 1: cabeza ===")
early_stop = EarlyStopping(monitor='val_loss', patience=10, min_delta=0.005,
                           restore_best_weights=True)

history_head = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

acc_head = model.evaluate(X_test, y_test, verbose=0)[1]
print(f"Test accuracy (solo cabeza): {acc_head:.4f}")

# ── Fase 2: fine-tuning del backbone ─────────────────────────────────────────
# Descongelamos el último 30% del backbone para ajuste fino
print("\n=== FASE 2: fine-tuning ===")
base.trainable = True
freeze_until = int(len(base.layers) * 0.70)
for layer in base.layers[:freeze_until]:
    layer.trainable = False

# LR muy pequeño para no destruir los pesos preentrenados
model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

history_ft = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop]
)

acc_ft = model.evaluate(X_test, y_test, verbose=0)[1]
print(f"Test accuracy (fine-tuning): {acc_ft:.4f}")

os.makedirs('../models', exist_ok=True)
model.save('../models/cnn_model.keras')
print("Modelo guardado en ../models/cnn_model.keras")

# ── Curvas de entrenamiento (ambas fases) ─────────────────────────────────────
n_head = len(history_head.history['accuracy'])
epochs_ft = range(n_head, n_head + len(history_ft.history['accuracy']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_head.history['accuracy'],     label='Train (cabeza)')
ax1.plot(history_head.history['val_accuracy'], label='Val (cabeza)')
ax1.plot(epochs_ft, history_ft.history['accuracy'],     '--', label='Train (fine-tuning)')
ax1.plot(epochs_ft, history_ft.history['val_accuracy'], '--', label='Val (fine-tuning)')
ax1.axvline(n_head - 1, color='gray', linestyle=':', label='inicio fine-tuning')
ax1.set_title('Accuracy — Transfer Learning'); ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy'); ax1.legend()

ax2.plot(history_head.history['loss'],     label='Train (cabeza)')
ax2.plot(history_head.history['val_loss'], label='Val (cabeza)')
ax2.plot(epochs_ft, history_ft.history['loss'],     '--', label='Train (fine-tuning)')
ax2.plot(epochs_ft, history_ft.history['val_loss'], '--', label='Val (fine-tuning)')
ax2.axvline(n_head - 1, color='gray', linestyle=':', label='inicio fine-tuning')
ax2.set_title('Loss — Transfer Learning'); ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss'); ax2.legend()

plt.tight_layout()
plt.show()

## Evaluación detallada por clase

El accuracy global no es suficiente en diagnóstico médico. Aquí vemos cuánto acierta el modelo en **cada tipo de lesión** por separado, y lo comparamos con el **baseline ZeroR** (lo que conseguiría un modelo que predice siempre la clase más frecuente, sin aprender nada).

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predicciones sobre el conjunto de test
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Baseline ZeroR: accuracy de predecir siempre la clase más frecuente
from collections import Counter
most_common = Counter(y_true).most_common(1)[0][1]
baseline = most_common / len(y_true)
print(f'Baseline ZeroR (predecir siempre la clase más frecuente): {baseline:.2%}')
print(f'Accuracy del modelo: {(y_pred == y_true).mean():.2%}')
print(f'Mejora sobre baseline: {(y_pred == y_true).mean() - baseline:+.2%}')

# Desglose por clase
print('Resultados por clase diagnóstica:')
print(classification_report(y_true, y_pred, target_names=le.classes_))

## Curvas ROC y matriz de confusión

La **curva ROC** mide cómo de bien el modelo separa cada clase del resto. El **AUC** (área bajo la curva) resume esto en un número: 1.0 = perfecto, 0.5 = aleatorio (no aprende nada).

En diagnóstico médico el AUC del **melanoma** (`mel`) es especialmente crítico: un falso negativo (diagnosticar como benigno algo maligno) tiene consecuencias graves.

La **matriz de confusión** muestra dónde se equivoca el modelo: qué clases confunde entre sí.

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import label_binarize

# ── Curvas ROC por clase ──────────────────────────────────────────────────────
y_true_bin = label_binarize(y_true, classes=range(len(le.classes_)))

fig, ax = plt.subplots(figsize=(9, 6))
for i, (clase, color) in enumerate(zip(le.classes_, plt.cm.tab10.colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{clase} (AUC = {auc(fpr, tpr):.2f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatorio (AUC = 0.50)')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Sensibilidad (Recall / TPR)')
ax.set_title('Curvas ROC por clase — CNN Transfer Learning')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

macro_auc = roc_auc_score(y_true_bin, y_pred_probs, multi_class='ovr', average='macro')
print(f'AUC macro (promedio One-vs-Rest): {macro_auc:.4f}')

# ── Matriz de confusión ───────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de confusión — CNN Transfer Learning')
plt.tight_layout()
plt.show()

## Conclusiones — CNN con Transfer Learning

El modelo de dos fases (cabeza + fine-tuning) mejora significativamente respecto a la CNN entrenada desde cero:

| Modelo | Accuracy |
|---|---|
| CNN desde cero (28×28) | ~74% |
| Transfer Learning (fase 1, cabeza) | ~82-85% |
| Transfer Learning (fase 2, fine-tuning) | ~85-90% |

**¿Por qué mejora tanto?**
- MobileNetV3Large aprendió a detectar bordes, texturas y estructuras complejas en 14M imágenes. Esas features generales son útiles para dermatología aunque el dominio sea diferente.
- El `Resizing(28→224)` dentro del modelo permite que el backbone opere a su resolución óptima sin necesidad de cambiar el dataset.
- La estrategia de dos fases evita destruir los pesos preentrenados (fase 1 estabiliza la cabeza antes de tocar el backbone).

**Limitación:** las imágenes del dataset son 28×28 originalmente — el upscale introduce interpolación artificial. Con imágenes de mayor resolución (≥96×96) el modelo podría mejorar aún más.